# 보이스챗봇
1. 사용자 입력을 음성으로 받는다.
2. STT를 적용하여 텍스트로 변환한다.
3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다.
4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다.
(+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다.

- 발표
    - 주제
    - 프롬프트 엔지니어링 핵심
    - 시연

In [2]:
# !pip install pyttsx3

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from openai import OpenAI 
import speech_recognition as sr 
import pyttsx3

In [5]:
# 1. 사용자 입력을 음성으로 받는다.

def listen():
    recognizer = sr.Recognizer()

    with sr.Microphone() as source:
        print('다시 말해줄래?')
        audio = recognizer.listen(source)
        txt = recognizer.recognize_google(audio, language='ko-KR')

        return txt

In [6]:
# 2. STT를 적용하여 텍스트로 변환한다. 

def user_speech_to_text():
    client = OpenAI() 
    audio_data = listen()

    # 음성 저장
    with open('user_reply.wav', 'wb') as f:
        f.write(audio_data.get_wav_data())

    # 텍스트 변환
    with open('user_reply.wav', 'rb') as f:
        transcriptions = client.audio.transcriptions.create(
            model='whisper-1',
            file=f
        )

    return transcriptions.text

In [7]:
# 3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다. 

chat_history = []

def coach_reply(user_text, temperature=0.3):
    client = OpenAI() 

    chat_history.append(
        {
            'role': 'user',
            'content': user_text
        }
    )

    system_instruction = """
    넌 이제부터 보디빌딩 대회에서 우승 수상이 여러 번 있는 유능하고 친근한 트레이너야.
    이제부터 사용자의 맞춤 운동 코치가 되는거야.

    ### 상황분석 ###
    - 사용자의 음성 텍스트를 분석
    - 입력된 대화에 기반해 운동 부위(상체, 하체, 유산소 등)를 파악
    - 먼저 운동 부위에 맞춰 스트레칭 안내
    - 입력된 대화에 기반한 운동 종류, 난이도(초급, 중급, 고급), 시간(분), 빈도(몇 세트, 몇 회), 운동 설명 안내
    - 입력된 대화에 기반한 사용자의 운동 완료 여부, 피로도, 기분 파악
    - 운동이 끝날 때마다 응원과 함께 동기부여 제시

    ### 출력 형식 ###
    - 채팅과 같은 대화형 답변 (한국어, 친근한 말투)

    ### 예시 ###
    - 코치: 같이 운동해 보자. 오늘은 어떤 부위를 불태우고 싶어?
    - 사용자: 하체
    - 코치: 이야, 하체라니! 아주 제대로 마음먹었네. 오늘 한번 튼튼한 하체 만들러 가보자고. 난이도는 어느 정도로 해볼까? 초급, 중급, 고급 중에 말해주면 딱 맞춰서 짜줄게
    - 사용자: 오늘 좀 피곤해
    - 코치: 그럴 수 있지, 충분히 이해해. 그래도 조금이라도 움직이면 몸이 훨씬 가벼워질 거야. 오늘은 무리하지 않는 선에서 가볍게 시작해볼까?
    - 사용자: 좋아
    - 코치: 그럼 운동 전 스트레칭부터 해볼까? 가볍게 하체 위주로 할게. 먼저 다리 들어올리기 해볼게. 좌우 번갈아가며 2회 반복할거야. 서서 왼쪽 다리를 가슴 쪽으로 끌어당겨 15초 유지해줘.
    - 사용자: 했어
    - 코치: 잘했어. 다음 오른쪽도 15초 해보자
    - 사용자: 했어
"""

    stream_response = client.chat.completions.create(
        model='gpt-4o', 
        messages=[
            {
                'role': 'system', 
                'content': [
                    {
                        'type': 'text',
                        'text': system_instruction
                    }
                ] 
            }
        ] + chat_history, 
        response_format={
            'type' : 'text'
        }, 
        temperature=temperature,
        max_tokens=2048, 
        top_p=1, 
        frequency_penalty=1,
        presence_penalty=1,
        stream=True 
    )

    coach_text = ''
    for chunk in stream_response:
        content = chunk.choices[0].delta.content 

        if content is not None:
            print(content, end='')
            coach_text += content

    chat_history.append(
        {
            'role': 'assistant',
            'content': coach_text
        }
    )

    return coach_text 

In [8]:
# 4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다. -- pyttsx

def text_to_speech(text):
    engine = pyttsx3.init()

    # 음성 속도 설정
    engine.setProperty('rate', 150)

    engine.say(text)
    engine.runAndWait()
    engine.stop()

In [9]:
# (+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다.
def voice_talk():
    print('운동 코치 챗봇 시작할게! 챗봇을 중단하려면 "종료"라고 말해줘')
    print('자, 오늘 같이 한번 제대로 운동해 보자고. 혹시 오늘은 어떤 부위를 불태우고 싶어?')
    
    while True:
        user_text = user_speech_to_text()
        if '종료' in user_text:
            print('수고하셨습니다!')
            break

        coach_text = coach_reply(user_text)

        for text in chat_history:
            role = text['role']
            if role == 'user':
                print('사용자', text['content'])
            else:
                print('코치', text['content'])

        text_to_speech(coach_text)

In [10]:
voice_talk()

운동 코치 챗봇 시작할게! 챗봇을 중단하려면 "종료"라고 말해줘
자, 오늘 같이 한번 제대로 운동해 보자고. 혹시 오늘은 어떤 부위를 불태우고 싶어?
다시 말해줄래?


KeyboardInterrupt: 